# ArthoBodh: train the BanglaBERT WSD model on Colab

**All you need is one file:** `data/processed/dataset_splits.json` from your project (about 4.5 MB).
The model code is included below. It's an exact copy of the project's `src/` files, so the trained model
works with your local web server as-is.

**Before you start:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

| Step | What happens |
|---|---|
| 0 | Check the GPU, install the Bengali normalizer |
| 1 | Upload `dataset_splits.json` |
| 2 | Write the model code (`config`, `text`, `model`, `evaluate`, `train`) |
| 3 | Look at what the model actually receives |
| 4 | Train |
| 5 | Evaluate on the 690 held-out test paragraphs |
| 6 | Try your own sentences |
| 7 | Download the trained model |

## 0. GPU check and install

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - Runtime > Change runtime type > T4 GPU")
!pip install -q git+https://github.com/csebuetnlp/normalizer

## 1. Upload the data
Choose `ArthoBodh/data/processed/dataset_splits.json` from your computer.
It contains the train / validation / test paragraphs and the definition of every meaning.

In [ ]:
import os, json, shutil
os.makedirs("/content/arthobodh/data/processed", exist_ok=True)
os.makedirs("/content/arthobodh/src", exist_ok=True)
%cd /content/arthobodh

from google.colab import files
uploaded = files.upload()
name = next(iter(uploaded))
shutil.move(name, "data/processed/dataset_splits.json")

d = json.load(open("data/processed/dataset_splits.json", encoding="utf-8"))
print(f"words: {len(d['catalog'])} | train: {len(d['train'])} | val: {len(d['val'])} | test: {len(d['test'])}")

## 2. Model code
Each cell below saves one file into `src/`. Read them in this order:

- **config.py**: paths and hyperparameters
- **text.py**: normalize the text, mark the target word with quotes, build one (context, `word : definition`) pair per meaning
- **model.py**: BanglaBERT scores each pair → the scores are grouped per word → softmax gives probabilities
- **evaluate.py**: inference on the validation/test sets, and the metrics
- **train.py**: cross-entropy over the meanings, AdamW, validation after each epoch, keep the best epoch, early stopping

In [ ]:
open('src/__init__.py', 'w').close()

In [ ]:
%%writefile src/config.py
"""
Central paths and hyperparameters, shared by training, evaluation and the web server.
"""

from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent

RAW_DATA_DIR = ROOT / "data" / "raw" / "Bengali_WSD_Database"
SPLITS_PATH = ROOT / "data" / "processed" / "dataset_splits.json"

CHECKPOINT_DIR = ROOT / "checkpoints" / "banglabert-wsd"
METRICS_DIR = ROOT / "results" / "metrics"
PLOTS_DIR = ROOT / "results" / "plots"

# Pretrained Bengali encoder (ELECTRA-base discriminator trained on Bengali text)
BASE_MODEL = "csebuetnlp/banglabert"

# Longest context in the dataset is 249 tokens; the gloss adds ~10-20 more.
MAX_LEN = 288

SEED = 42
BATCH_SIZE = 8          # training examples per step; each expands to 3-4 (context, gloss) pairs
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_EPOCHS = 10
PATIENCE = 3            # stop after this many epochs without validation improvement

In [ ]:
%%writefile src/text.py
"""
Turns (context, target word, candidate senses) into model inputs.

Training, evaluation and the web server all go through build_pairs(), so the model
sees exactly the same input format everywhere.

For one context and a word with N senses we create N text pairs:

    first  segment:  the context, with the target word wrapped in " quotes "
    second segment:  "<target word> : <sense definition>"

BanglaBERT reads each pair jointly and scores how well that definition fits the
marked word in that context.
"""

import re

try:
    from normalizer import normalize as _bn_normalize   # csebuetnlp normalizer used when BanglaBERT was pretrained
except ImportError:                                      # pragma: no cover
    _bn_normalize = None

_ZERO_WIDTH = re.compile(r"[​‌‍﻿]")
_SPACES = re.compile(r"\s+")
_BENGALI_CHAR = r"ঀ-৿"


def normalize_text(text: str) -> str:
    text = str(text)
    if _bn_normalize is not None:
        text = _bn_normalize(text)
    text = _ZERO_WIDTH.sub("", text)
    return _SPACES.sub(" ", text).strip()


def mark_target(context: str, target: str) -> tuple[str, bool]:
    """
    Wraps every occurrence of the target that starts a word (so জল matches জল and জলের,
    but not the middle of another word) in quotes. Returns (marked_text, found).
    """
    pattern = re.compile(rf"(?<![{_BENGALI_CHAR}])({re.escape(target)}[{_BENGALI_CHAR}]*)")
    marked, count = pattern.subn(r'" \1 "', context)
    return marked, count > 0


def sorted_senses(senses: dict) -> list[tuple[int, str]]:
    """Catalog senses as [(sense_num, definition), ...] ordered by sense number."""
    return sorted(((int(k), v) for k, v in senses.items()), key=lambda kv: kv[0])


def build_pairs(context: str, target: str, senses: dict):
    """
    Returns (first_segments, second_segments, sense_nums, target_found).
    One entry per candidate sense, in sense-number order.
    """
    target = normalize_text(target)
    marked, found = mark_target(normalize_text(context), target)
    firsts, seconds, nums = [], [], []
    for num, definition in sorted_senses(senses):
        firsts.append(marked)
        seconds.append(f"{target} : {normalize_text(definition)}")
        nums.append(num)
    return firsts, seconds, nums, found

In [ ]:
%%writefile src/model.py
"""
Gloss-matching cross-encoder for Bengali WSD.

    pair (marked context, "word : definition")
        -> BanglaBERT encoder
        -> [CLS] vector
        -> small classification head
        -> one real-valued match score

All candidate senses of a word are scored this way, the scores are put side by side,
and a softmax over them gives the probability of each sense. Training minimises
cross-entropy against the correct sense, which pushes the right definition's score up
and the other definitions' scores down for that context.

Because the definition text is part of the input, the model learns what it means for
a context to *match a meaning*, instead of memorising "slot 2 of word 57".
"""

import json
from pathlib import Path

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from . import config
from .text import build_pairs


class GlossWSDModel:
    def __init__(self, encoder, tokenizer, device=None):
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.encoder = encoder.to(self.device)
        self.tokenizer = tokenizer

    # ---- construction / persistence -------------------------------------------------

    @classmethod
    def from_pretrained_base(cls, base_model=config.BASE_MODEL, device=None):
        """Fresh model for training: pretrained BanglaBERT + untrained 1-output scoring head."""
        tokenizer = AutoTokenizer.from_pretrained(base_model)
        encoder = AutoModelForSequenceClassification.from_pretrained(base_model, num_labels=1)
        return cls(encoder, tokenizer, device)

    @classmethod
    def load(cls, checkpoint_dir=config.CHECKPOINT_DIR, device=None):
        """Fine-tuned model saved by save()."""
        checkpoint_dir = Path(checkpoint_dir)
        tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
        encoder = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
        model = cls(encoder, tokenizer, device)
        model.encoder.eval()
        return model

    def save(self, checkpoint_dir=config.CHECKPOINT_DIR, info=None):
        checkpoint_dir = Path(checkpoint_dir)
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.encoder.save_pretrained(checkpoint_dir)
        self.tokenizer.save_pretrained(checkpoint_dir)
        if info is not None:
            with open(checkpoint_dir / "training_info.json", "w", encoding="utf-8") as f:
                json.dump(info, f, ensure_ascii=False, indent=2)

    # ---- scoring --------------------------------------------------------------------

    def encode(self, items):
        """
        items: list of (context, target_word, senses_dict).
        Returns tokenized pairs for every candidate of every item, plus where each pair
        belongs in the [num_items, max_senses] score matrix.
        """
        firsts, seconds, rows, cols, sense_nums = [], [], [], [], []
        for row, (context, target, senses) in enumerate(items):
            f, s, nums, _ = build_pairs(context, target, senses)
            firsts += f
            seconds += s
            rows += [row] * len(nums)
            cols += list(range(len(nums)))
            sense_nums.append(nums)

        enc = self.tokenizer(
            firsts, seconds,
            max_length=config.MAX_LEN,
            truncation="only_first",       # never cut the definition
            padding=True,
            return_tensors="pt",
        )
        return enc, torch.tensor(rows), torch.tensor(cols), sense_nums

    def score(self, enc, rows, cols, num_items):
        """Forward pass -> [num_items, max_senses] logits; missing senses are -inf."""
        enc = {k: v.to(self.device) for k, v in enc.items()}
        pair_scores = self.encoder(**enc).logits.squeeze(-1).float()
        max_senses = int(cols.max().item()) + 1
        matrix = torch.full((num_items, max_senses), float("-inf"), device=self.device)
        matrix[rows.to(self.device), cols.to(self.device)] = pair_scores
        return matrix

    @torch.no_grad()
    def predict(self, context, target_word, senses):
        """
        Returns a list of {"sense_num", "sense", "probability"} sorted by probability,
        plus whether the target word was found in the context.
        """
        self.encoder.eval()
        _, _, _, found = build_pairs(context, target_word, senses)
        enc, rows, cols, sense_nums = self.encode([(context, target_word, senses)])
        probs = torch.softmax(self.score(enc, rows, cols, 1), dim=-1)[0].cpu().tolist()
        defs = {int(k): v for k, v in senses.items()}
        ranked = [
            {"sense_num": num, "sense": defs[num], "probability": probs[i]}
            for i, num in enumerate(sense_nums[0])
        ]
        ranked.sort(key=lambda r: -r["probability"])
        return ranked, found

In [ ]:
%%writefile src/evaluate.py
"""
Evaluates a fine-tuned checkpoint on the held-out test split.

    python -m src.evaluate                    # uses checkpoints/banglabert-wsd
    python -m src.evaluate --checkpoint PATH

Writes results/metrics/transformer_metrics.json, transformer_test_predictions.json and
results/plots/transformer_confusion_matrix.png. Metrics are computed on the relative
sense label (sense 1..4), the same way as the TF-IDF + SVM baseline, so the numbers
are directly comparable.
"""

import argparse
import json
import sys
from collections import defaultdict

import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

from . import config

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")


def load_splits():
    with open(config.SPLITS_PATH, "r", encoding="utf-8") as f:
        return json.load(f)


def label_of(record, catalog):
    """Index of the record's true sense among its word's senses (sorted by sense number)."""
    nums = sorted(int(k) for k in catalog[record["folder"]]["senses"])
    return nums.index(record["sense_num"])


@torch.no_grad()
def run_inference(model, records, catalog, batch_size=config.EVAL_BATCH_SIZE):
    """Returns (predicted label indices, probability rows, mean loss)."""
    model.encoder.eval()
    preds, prob_rows, total_loss = [], [], 0.0
    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]
        items = [(r["text"], r["target_word"], catalog[r["folder"]]["senses"]) for r in batch]
        labels = torch.tensor([label_of(r, catalog) for r in batch], device=model.device)
        enc, rows, cols, _ = model.encode(items)
        logits = model.score(enc, rows, cols, len(batch))
        total_loss += F.cross_entropy(logits, labels, reduction="sum").item()
        probs = torch.softmax(logits, dim=-1)
        preds += probs.argmax(dim=-1).tolist()
        prob_rows += probs.cpu().tolist()
    return preds, prob_rows, total_loss / max(1, len(records))


def compute_metrics(y_true, y_pred, num_senses):
    def block(t, p):
        if not t:
            return None
        _, _, f1_macro, _ = precision_recall_fscore_support(t, p, average="macro", zero_division=0)
        return {"instances": len(t), "accuracy": accuracy_score(t, p), "macro_f1": f1_macro}

    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    _, _, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    three = [(t, p) for t, p, n in zip(y_true, y_pred, num_senses) if n == 3]
    four = [(t, p) for t, p, n in zip(y_true, y_pred, num_senses) if n == 4]
    return {
        "test_instances": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": prec,
        "macro_recall": rec,
        "macro_f1": f1,
        "weighted_f1": f1_w,
        "three_senses": block([t for t, _ in three], [p for _, p in three]),
        "four_senses": block([t for t, _ in four], [p for _, p in four]),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3]).tolist(),
    }


def save_confusion_matrix(cm, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import numpy as np

    cm = np.array(cm)
    fig, ax = plt.subplots(figsize=(6, 5), dpi=150)
    im = ax.imshow(cm, cmap=plt.cm.Greens)
    fig.colorbar(im, ax=ax)
    names = ["Sense 1", "Sense 2", "Sense 3", "Sense 4"]
    ax.set(xticks=range(4), yticks=range(4), xticklabels=names, yticklabels=names,
           title="BanglaBERT gloss cross-encoder: confusion matrix",
           ylabel="True sense", xlabel="Predicted sense")
    for i in range(4):
        for j in range(4):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontweight="bold",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def main():
    from .model import GlossWSDModel

    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", default=str(config.CHECKPOINT_DIR))
    parser.add_argument("--limit", type=int, default=None, help="evaluate only the first N test records (smoke test)")
    args = parser.parse_args()

    data = load_splits()
    catalog, test = data["catalog"], data["test"][:args.limit]

    model = GlossWSDModel.load(args.checkpoint)
    print(f"Evaluating {args.checkpoint} on {len(test)} test contexts ({model.device})...")
    preds, probs, loss = run_inference(model, test, catalog)

    y_true = [label_of(r, catalog) for r in test]
    num_senses = [len(catalog[r["folder"]]["senses"]) for r in test]
    metrics = {"model": "BanglaBERT gloss cross-encoder", "test_loss": loss,
               **compute_metrics(y_true, preds, num_senses)}

    per_word = defaultdict(lambda: [0, 0])
    predictions = []
    for r, t, p, pr in zip(test, y_true, preds, probs):
        senses = catalog[r["folder"]]["senses"]
        nums = sorted(int(k) for k in senses)
        per_word[r["target_word"]][0] += int(t == p)
        per_word[r["target_word"]][1] += 1
        predictions.append({
            "folder": r["folder"], "target_word": r["target_word"], "context": r["text"],
            "true_sense_num": nums[t], "true_sense": senses[str(nums[t])],
            "predicted_sense_num": nums[p], "predicted_sense": senses[str(nums[p])],
            "confidence": pr[p], "correct": t == p,
        })
    metrics["per_word_accuracy"] = {w: c / n for w, (c, n) in sorted(per_word.items(), key=lambda kv: kv[1][0] / kv[1][1])}

    config.METRICS_DIR.mkdir(parents=True, exist_ok=True)
    config.PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    with open(config.METRICS_DIR / "transformer_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)
    with open(config.METRICS_DIR / "transformer_test_predictions.json", "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)
    save_confusion_matrix(metrics["confusion_matrix"], config.PLOTS_DIR / "transformer_confusion_matrix.png")

    print("\n================ TEST RESULTS ================")
    print(f"Accuracy:     {metrics['accuracy'] * 100:.2f}%")
    print(f"Macro F1:     {metrics['macro_f1'] * 100:.2f}%")
    print(f"Weighted F1:  {metrics['weighted_f1'] * 100:.2f}%")
    for key, label in (("three_senses", "3-sense words"), ("four_senses", "4-sense words")):
        if metrics[key]:
            print(f"  {label}: accuracy {metrics[key]['accuracy'] * 100:.2f}% (n={metrics[key]['instances']})")
    baseline_path = config.METRICS_DIR / "baseline_svm_metrics.json"
    if baseline_path.exists():
        with open(baseline_path, encoding="utf-8") as f:
            base = json.load(f)
        print(f"TF-IDF + SVM baseline accuracy: {base['accuracy'] * 100:.2f}%")
    hardest = list(metrics["per_word_accuracy"].items())[:5]
    print("Hardest words:", ", ".join(f"{w} {a * 100:.0f}%" for w, a in hardest))
    print("==============================================")
    print(f"Saved metrics and predictions to {config.METRICS_DIR}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/train.py
"""
Fine-tunes BanglaBERT as a gloss-matching cross-encoder.

    python -m src.train                       # full training (use a GPU, e.g. Colab T4)
    python -m src.train --limit 16 --epochs 1 # quick CPU smoke test

Each epoch:
  1. For every training context, score all candidate senses of its word.
  2. Cross-entropy over those scores against the correct sense; update the model.
  3. Measure accuracy on the validation split.
  4. If validation accuracy improved, save the model to checkpoints/banglabert-wsd.
Training stops early after PATIENCE epochs without improvement.
"""

import argparse
import json
import random
import sys
import time

import numpy as np
import torch
import torch.nn.functional as F
from transformers import get_linear_schedule_with_warmup

from . import config
from .evaluate import label_of, load_splits, run_inference
from .model import GlossWSDModel

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=config.MAX_EPOCHS)
    parser.add_argument("--lr", type=float, default=config.LEARNING_RATE)
    parser.add_argument("--batch-size", type=int, default=config.BATCH_SIZE)
    parser.add_argument("--output", default=str(config.CHECKPOINT_DIR))
    parser.add_argument("--limit", type=int, default=None, help="use only N train/val records (smoke test)")
    args = parser.parse_args()

    set_seed(config.SEED)
    data = load_splits()
    catalog = data["catalog"]
    train, val = data["train"], data["val"]
    if args.limit:
        train, val = train[:args.limit], val[:args.limit]

    model = GlossWSDModel.from_pretrained_base()
    device = model.device
    use_amp = device.type == "cuda"

    optimizer = torch.optim.AdamW(model.encoder.parameters(), lr=args.lr, weight_decay=config.WEIGHT_DECAY)
    steps_per_epoch = (len(train) + args.batch_size - 1) // args.batch_size
    total_steps = steps_per_epoch * args.epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * config.WARMUP_RATIO), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    print(f"Base model: {config.BASE_MODEL} | device: {device} | mixed precision: {use_amp}")
    print(f"Train contexts: {len(train)} | Val contexts: {len(val)} | "
          f"batch: {args.batch_size} | lr: {args.lr} | max epochs: {args.epochs}")

    best_acc, best_loss, bad_epochs, history = -1.0, float("inf"), 0, []
    for epoch in range(1, args.epochs + 1):
        t0 = time.time()
        model.encoder.train()
        order = list(range(len(train)))
        random.shuffle(order)
        running_loss, correct = 0.0, 0

        for step, start in enumerate(range(0, len(order), args.batch_size), 1):
            batch = [train[i] for i in order[start:start + args.batch_size]]
            items = [(r["text"], r["target_word"], catalog[r["folder"]]["senses"]) for r in batch]
            labels = torch.tensor([label_of(r, catalog) for r in batch], device=device)
            enc, rows, cols, _ = model.encode(items)

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model.score(enc, rows, cols, len(batch))   # [batch, max_senses]
            loss = F.cross_entropy(logits, labels)

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.encoder.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running_loss += loss.item() * len(batch)
            correct += (logits.argmax(dim=-1) == labels).sum().item()
            if step % 50 == 0:
                print(f"  epoch {epoch} step {step}/{steps_per_epoch} loss {running_loss / (start + len(batch)):.4f}")

        train_loss, train_acc = running_loss / len(train), correct / len(train)
        val_preds, _, val_loss = run_inference(model, val, catalog)
        val_acc = float(np.mean([p == label_of(r, catalog) for p, r in zip(val_preds, val)]))
        history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                        "val_loss": val_loss, "val_acc": val_acc})
        print(f"Epoch {epoch} ({time.time() - t0:.0f}s) | train loss {train_loss:.4f} acc {train_acc * 100:.2f}% "
              f"| val loss {val_loss:.4f} acc {val_acc * 100:.2f}%")

        # Best = highest validation accuracy; ties broken by lower validation loss.
        if val_acc > best_acc or (val_acc == best_acc and val_loss < best_loss):
            best_acc, best_loss, bad_epochs = val_acc, val_loss, 0
            model.save(args.output, info={"base_model": config.BASE_MODEL, "best_epoch": epoch,
                                          "val_accuracy": val_acc, "val_loss": val_loss,
                                          "max_len": config.MAX_LEN, "history": history})
            print(f"  -> saved best model to {args.output}")
        else:
            bad_epochs += 1
            if bad_epochs >= config.PATIENCE:
                print(f"No validation improvement for {config.PATIENCE} epochs; stopping.")
                break

    config.METRICS_DIR.mkdir(parents=True, exist_ok=True)
    with open(config.METRICS_DIR / "transformer_training_history.json", "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)
    print(f"Best validation accuracy: {best_acc * 100:.2f}%. Next: python -m src.evaluate")


if __name__ == "__main__":
    main()

## 3. What the model actually sees
One training paragraph becomes one pair per meaning of its word. BanglaBERT scores every pair; the correct pair should get the highest score.

In [ ]:
from src.text import build_pairs
r = d["train"][0]
senses = d["catalog"][r["folder"]]["senses"]
firsts, seconds, nums, found = build_pairs(r["text"], r["target_word"], senses)
print("target word:", r["target_word"], "| correct meaning:", r["sense_num"], "-", r["sense_def"])
print("context (A):", firsts[0][:200], "...")
print()
for n, s in zip(nums, seconds):
    print(f"pair {n}: B = {s}", "  <-- correct" if n == r["sense_num"] else "")

## 4. Train
Takes about 2–4 minutes per epoch on a T4. The best epoch (by validation accuracy) is saved to `checkpoints/banglabert-wsd`.

In [ ]:
!python -m src.train

In [ ]:
import matplotlib.pyplot as plt
h = json.load(open("results/metrics/transformer_training_history.json"))
ep = [x["epoch"] for x in h]
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
a.plot(ep, [x["train_loss"] for x in h], "o-", label="train"); a.plot(ep, [x["val_loss"] for x in h], "s-", label="val"); a.set_title("Loss"); a.set_xlabel("epoch"); a.legend()
b.plot(ep, [x["train_acc"] for x in h], "o-", label="train"); b.plot(ep, [x["val_acc"] for x in h], "s-", label="val"); b.set_title("Accuracy"); b.set_xlabel("epoch"); b.legend()
os.makedirs("results/plots", exist_ok=True)
plt.savefig("results/plots/transformer_training_curves.png", dpi=150, bbox_inches="tight"); plt.show()

## 5. Evaluate on the test set
The test set has 690 paragraphs that were never used in training or model selection. The TF-IDF + SVM baseline scored **69.42%**.

In [ ]:
!python -m src.evaluate

In [ ]:
from IPython.display import Image, display
display(Image("results/plots/transformer_confusion_matrix.png", width=450))
m = json.load(open("results/metrics/transformer_metrics.json", encoding="utf-8"))
print("Hardest words:")
for w, acc in list(m["per_word_accuracy"].items())[:10]:
    print(f"  {w}: {acc*100:.0f}%")
preds = json.load(open("results/metrics/transformer_test_predictions.json", encoding="utf-8"))
print()
print("Some mistakes:")
for p in [p for p in preds if not p["correct"]][:5]:
    print(f"  [{p['target_word']}] true: {p['true_sense']} | predicted: {p['predicted_sense']} ({p['confidence']*100:.0f}%)")
    print("     ", p["context"][:120], "...")

## 6. Try your own sentences

In [ ]:
from src.model import GlossWSDModel
model = GlossWSDModel.load()
catalog = {info["target_word"]: info["senses"] for info in d["catalog"].values()}

def explain(sentence, word):
    ranked, found = model.predict(sentence, word, catalog[word])
    print(sentence, "| target:", word, "" if found else "(word not found in sentence!)")
    for r in ranked:
        print(f"   {r['probability']*100:5.1f}%  {r['sense']}")
    print()

print("Supported words:", ", ".join(sorted(catalog)))
print()
explain("বৃষ্টির জল জমে রাস্তা ডুবে গেছে।", "জল")
explain("মেয়েটির চোখে জল এসে গেল।", "জল")

## 7. Download the trained model
Unzip `arthobodh_model.zip` **inside your local `ArthoBodh` folder**. You'll get `checkpoints/banglabert-wsd/`
and the new files in `results/`. Then run `run_server.bat`.

The zip is about 400 MB. If the browser download fails, use the Google Drive cell below instead.

In [ ]:
!zip -qr arthobodh_model.zip checkpoints results
!ls -lh arthobodh_model.zip
files.download("arthobodh_model.zip")

In [ ]:
# Optional: save to Google Drive instead (more reliable for large files)
from google.colab import drive
drive.mount("/content/drive")
!cp arthobodh_model.zip /content/drive/MyDrive/
print("Saved to MyDrive/arthobodh_model.zip")